In [2]:
import pandas as pd

In [3]:
from sqlalchemy import create_engine

SQLAlchemy is the bridge/tool, and SQLite is the actual database it's writing into

CSV is currently just flat, unstructured text-with-commas — no real "table" structure, no data types enforced, no ability to run SQL queries on it directly

Pandas reads that CSV into a DataFrame (an in-memory table-like structure)
SQLAlchemy (via pandas' .to_sql() method) takes that DataFrame and writes it into an actual SQLite database file (.db) — creating a real table with defined columns, proper rows, and now it's genuinely queryable using standard SQL

SQLite stores data in structured rows/columns/tables — and it's a serverless, file-based database (the whole database lives in one .db file on your computer, no separate database server needed to run) — which is exactly why it's the simplest choice for a personal project like this, versus something like MySQL or PostgreSQL which need an actual running server.

In [4]:
engine=create_engine('sqlite:///food_delivery.db')

In [5]:
df=pd.read_csv(r'D:\Personal_Projects\Food_Delivery_SQL_Project\data\swiggy_vs_zomato_3000.csv')

In [6]:
df.to_sql('restaurants', engine, if_exists='replace', index=False)
print(f"Loaded {len(df)} rows into 'restaurants' table inside food_delivery.db")
print(f"Columns: {len(df.columns)}")

Loaded 2500 rows into 'restaurants' table inside food_delivery.db
Columns: 41


How many restaurants are there in each city?

In [7]:
query_1="SELECT city,COUNT(*) AS restaurant_count FROM restaurants GROUP BY city ORDER BY COUNT(*) DESC"
pd.read_sql(query_1,engine)

,city,restaurant_count
0,Mumbai,450
1,Bangalore,420
2,Delhi,380
3,Pune,250
4,Hyderabad,240
5,Chennai,200
6,Kolkata,180
7,Ahmedabad,160
8,Jaipur,120
9,Lucknow,100


What is the average Swiggy rating and average Zomato rating, across all restaurants?

In [8]:
query_2="SELECT AVG(swiggy_rating) AS swiggy_avg_rating,AVG(zomato_rating) AS zomato_avg_rating FROM restaurants"
pd.read_sql(query_2,engine)

,swiggy_avg_rating,zomato_avg_rating
0,3.98564,4.006


which cuisine type appears most frequently in the dataset

In [9]:
df.columns

Index(['restaurant_id', 'restaurant_name', 'city', 'locality',
       'restaurant_type', 'cuisines', 'distance_from_city_center_km',
       'opening_time', 'closing_time', 'days_operational', 'swiggy_rating',
       'swiggy_total_reviews', 'zomato_rating', 'zomato_total_reviews',
       'average_rating_both_platforms', 'avg_cost_per_person_inr',
       'price_category', 'swiggy_delivery_fee_inr', 'zomato_delivery_fee_inr',
       'swiggy_avg_delivery_time_minutes', 'zomato_avg_delivery_time_minutes',
       'swiggy_platform_commission_pct', 'zomato_platform_commission_pct',
       'swiggy_discount_frequency_pct', 'zomato_discount_frequency_pct',
       'swiggy_estimated_monthly_orders', 'zomato_estimated_monthly_orders',
       'swiggy_estimated_monthly_revenue_inr',
       'zomato_estimated_monthly_revenue_inr',
       'swiggy_estimated_net_profit_inr', 'zomato_estimated_net_profit_inr',
       'swiggy_market_share_pct', 'zomato_market_share_pct',
       'platform_performance_better',

In [10]:
query_3="SELECT cuisines,COUNT(*) AS cuisine_across_rest FROM restaurants GROUP BY cuisines ORDER BY COUNT(*) DESC LIMIT 5"
pd.read_sql(query_3,engine)

,cuisines,cuisine_across_rest
0,Biryani,84
1,Street Food,80
2,Mediterranean,80
3,Korean,79
4,Bakery,79


What is the average cost per person (avg_cost_per_person_inr), grouped by price_category

In [11]:
query4="SELECT price_category,AVG(avg_cost_per_person_inr) AS avg_cost_per_person_per_category FROM restaurants GROUP BY price_category"
pd.read_sql(query4,engine)

,price_category,avg_cost_per_person_per_category
0,Budget,225.298361
1,Luxury,1345.775801
2,Mid-Range,497.381020
3,Premium,950.823085


: How many restaurants have has_own_app = True

In [12]:
query_5="SELECT COUNT(*) AS count_rest FROM restaurants WHERE has_own_app=1"
pd.read_sql(query_5,engine)

,count_rest
0,1284


Subqueries

Find restaurants whose avg_cost_per_person_inr is above the overall average. Show name, city, and cost.

In [15]:
query6="SELECT restaurant_name,city,avg_cost_per_person_inr FROM restaurants WHERE (avg_cost_per_person_inr) > (SELECT AVG(avg_cost_per_person_inr) FROM restaurants)"
pd.read_sql(query6,engine)

,restaurant_name,city,avg_cost_per_person_inr
0,Restaurant_1,Mumbai,1014
1,Restaurant_2,Mumbai,925
2,Restaurant_3,Mumbai,1450
3,Restaurant_5,Mumbai,1249
4,Restaurant_10,Mumbai,1220
...,...,...,...
1268,Restaurant_2496,Lucknow,891
1269,Restaurant_2497,Lucknow,1276
1270,Restaurant_2498,Lucknow,1083
1271,Restaurant_2499,Lucknow,1227


How many restaurants have Swiggy net profit above the average Swiggy net profit?

In [16]:
query7="SELECT COUNT(*) FROM restaurants WHERE swiggy_estimated_net_profit_inr > (SELECT AVG(swiggy_estimated_net_profit_inr) FROM restaurants)"
pd.read_sql(query7,engine)

,COUNT(*)
0,1098


what distinct values does platform_performance_better have, and how many restaurants fall under each

In [21]:
query8="SELECT platform_performance_better,COUNT(*) FROM restaurants GROUP BY platform_performance_better"
pd.read_sql(query8,engine)

,platform_performance_better,COUNT(*)
0,Balanced,32
1,Swiggy Better,1195
2,Zomato Better,1273


For each city, find the average average_rating_both_platforms, then list only cities that are above the overall average across all restaurants

In [26]:
query9="SELECT city,AVG(average_rating_both_platforms) FROM restaurants GROUP BY city HAVING average_rating_both_platforms>(SELECT AVG(average_rating_both_platforms) FROM restaurants)"
pd.read_sql(query9,engine)

,city,AVG(average_rating_both_platforms)
0,Ahmedabad,4.017188
1,Bangalore,4.000238
2,Chennai,3.985500
3,Delhi,3.976974
4,Hyderabad,3.978125
5,Jaipur,4.007917
6,Kolkata,3.990556
7,Pune,4.006400
